# Agreement Term Extraction
*Illustrative Example*

In [ ]:
import json
from collections import defaultdict

import pandas as pd
import streamlit as st
from snowflake.snowpark.context import get_active_session

from contract_extractor import ContractExtractor

session = get_active_session()


"""
Simple inline Streamlit viewer for contract extraction results.
Copy this code directly into your notebook after getting the results.
"""

def display_results(results):
    """Display contract extraction results inline."""

    # Document Info
    st.header("📄 Document Information")
    doc = results.get('document', {})
    col1, col2 = st.columns(2)
    with col1:
        st.write(f"**Title:** {doc.get('title', 'N/A')}")
        st.write(f"**Suncor ID:** `{doc.get('suncor_agrmnt_num', 'N/A')}`")
        st.write(f"**Counter Party ID:** `{doc.get('counter_party_agrmnt_num', 'N/A')}`")
    with col2:
        st.write(f"**Effective:** {doc.get('effective_start', 'N/A')} to {doc.get('effective_end', 'N/A')}")
        st.write(f"**Governing Law:** {doc.get('governing_law', 'N/A')}")

    st.markdown("---")

    # Parties
    st.header("🤝 Parties")
    parties = results.get('parties', [])
    cols = st.columns(len(parties))
    for idx, party in enumerate(parties):
        with cols[idx]:
            st.write(f"**{party.get('role')}:** {party.get('name')}")

    st.markdown("---")

    # Terms by Category
    st.header("📋 Contract Terms")
    terms = results.get('terms', [])

    # Group by category
    by_category = defaultdict(list)
    for term in terms:
        by_category[term.get('category', 'Other')].append(term)

    # Display each category
    for category, cat_terms in sorted(by_category.items()):
        st.subheader(category)
        with st.expander(f"{category} ({len(cat_terms)} terms)", expanded=False):
            for term in cat_terms:
                st.markdown(f"### `{term.get('term_id')}`")
                st.write(f"**{term.get('summary')}**")
                st.caption(f"Source: {term.get('source_location')}")

                # Key details
                cols = st.columns(3)
                with cols[0]:
                    if term.get('obligor'):
                        st.write(f"👤 Obligor: {term['obligor']}")
                with cols[1]:
                    if term.get('obligee'):
                        st.write(f"👥 Obligee: {term['obligee']}")
                with cols[2]:
                    st.write(f"✓ Confidence: {term.get('confidence', 0):.0%}")

                # Additional details (no nested expander)
                if term.get('quantitative_details'):
                    st.markdown("**📊 Details:**")
                    for k, v in term['quantitative_details'].items():
                        st.write(f"• {k.replace('_', ' ').title()}: {v}")

                if term.get('timeframe'):
                    st.write(f"⏰ {term['timeframe'].get('description', 'See timeframe details')}")

                if term.get('comments'):
                    st.write(f"📒 {term['comments']}")

                st.markdown("---")

def main():
    try:
        extractor = ContractExtractor(
        session=session,
        response_format_path='response_format.json',
        system_prompt_path='system_prompt.md'
        )
     
        # Extract contract terms
        results = extractor.extract_from_stage(
            stage_path='@POC_SUNCHAT_CORTEX.DEV.STO_CONTRACTS/123-737 Montreal-ON Product P&S Agmt - Jan 1, 2025',
            filename='123-737 Montreal-ON Product P&S Agmt_2025_01_01.pdf',
            model='claude-4-sonnet',
            temperature=0.0,
            parse_mode="OCR" # Can do LAYOUT but make take longer depending on queue
        )
    except:
        with open('results.json', 'r') as json_file:
            return json.load(json_file)

    return results

**Illustrative example** of using Snowflake Cortex AI to fill out a **pre-defined set of term structures to extract** from a given contract. 

The precise term structure, a.k.a. "terms set", will be refined based on requirements. Here, we extract agreement metadata such as effective dates AND precise term metadata, details and descriptions for further processing.

In [ ]:
st.image('architecture2.png')

In [ ]:
results = main()
display_results(results)

In [ ]:
LS '@POC_SUNCHAT_CORTEX.DEV.STO_CONTRACTS/123-737 Montreal-ON Product P&S Agmt - Jan 1, 2025/123-737 Montreal-ON Product P&S Agmt_2025_01_01.pdf';

In [ ]:
SELECT TO_VARCHAR(
            AI_PARSE_DOCUMENT(
                TO_FILE('@POC_SUNCHAT_CORTEX.DEV.STO_CONTRACTS/123-737 Montreal-ON Product P&S Agmt - Jan 1, 2025', '123-737 Montreal-ON Product P&S Agmt_2025_01_01.pdf'),
                OBJECT_CONSTRUCT('mode', 'LAYOUT')
            )
        ) AS parsed_content

In [ ]:
print(json.dumps(results, indent = 2))